<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/TIFA%20diagnosis%20v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

1. Save as tifa_diagnostic.py

2. Install dependencies if needed:
   pip install numpy scipy

3. Run:
   python tifa_diagnostic.py

4. Copy the full output
   and paste it here

SyntaxError: invalid syntax (2987970500.py, line 1)

In [1]:

"""
TIFA Diagnostic Code - Version 2
Fixed chi2 units
Brother Tony / Claude
"""

import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import curve_fit

# ============================================================
# FIXED COSMOLOGICAL PARAMETERS
# ============================================================
H0      = 67.36
OmegaM  = 0.315
OmegaDE = 0.685
MPl     = 1.0
rd      = 147.09
c_kms   = 299792.458
DH0     = c_kms / H0    # = 4451.5 Mpc

# ============================================================
# DESI DR1 BAO DATA
# type 0 = D_M/rd
# type 1 = D_H/rd
# ============================================================
DESI_data = [
    [0.295,   7.93,  0.15,  0],
    [0.295,  20.08,  0.60,  1],
    [0.510,  13.62,  0.25,  0],
    [0.510,  13.11,  0.42,  1],
    [0.706,  16.85,  0.32,  0],
    [0.706,  10.34,  0.28,  1],
    [0.930,  21.71,  0.28,  0],
    [0.930,   8.00,  0.17,  1],
    [1.317,  27.79,  0.69,  0],
    [1.317,   6.13,  0.13,  1],
    [1.491,  30.21,  0.79,  0],
    [1.491,   5.65,  0.13,  1],
    [2.330,  39.71,  0.94,  0],
]

# ============================================================
# CPL EQUATION OF STATE
# ============================================================
def w_cpl(a, w0, wa):
    return w0 + wa * (1.0 - a)

# ============================================================
# HUBBLE FUNCTION USING CPL
# ============================================================
def H_over_H0(a, w0, wa):
    """
    H(a)/H0 for flat LCDM + CPL dark energy.
    Exact integral form.
    """
    if a <= 0:
        return 1e10

    def integrand(ap):
        return (1.0 + w_cpl(ap, w0, wa)) / ap

    integral, _ = quad(integrand, a, 1.0,
                       limit=200,
                       epsabs=1e-10,
                       epsrel=1e-10)

    rho_de_ratio = np.exp(3.0 * integral)
    H2 = OmegaM / a**3 + OmegaDE * rho_de_ratio
    return np.sqrt(max(H2, 0.0))

# ============================================================
# COMOVING DISTANCE IN MPC
# ============================================================
def D_C_Mpc(z, w0, wa):
    """
    Comoving distance in Mpc.
    """
    def integrand(zp):
        a = 1.0 / (1.0 + zp)
        return 1.0 / H_over_H0(a, w0, wa)

    result, _ = quad(integrand, 0.0, z,
                     limit=200,
                     epsabs=1e-10,
                     epsrel=1e-10)

    return result * DH0   # convert to Mpc

# ============================================================
# CHI2 - CORRECTED UNITS
# ============================================================
def compute_chi2(w0, wa):
    """
    Chi2 against DESI DR1 BAO.
    All distances in Mpc.
    """
    chi2 = 0.0

    for row in DESI_data:
        z, obs, sigma, dtype = row
        a = 1.0 / (1.0 + z)

        if dtype == 0:
            # D_M / rd
            # flat universe: D_M = D_C
            DC = D_C_Mpc(z, w0, wa)
            theory = DC / rd

        else:
            # D_H / rd = (c / H(z)) / rd
            Hz = H0 * H_over_H0(a, w0, wa)
            DH = c_kms / Hz
            theory = DH / rd

        chi2 += ((obs - theory) / sigma) ** 2

    return chi2

# ============================================================
# TIFA FIELD SOLVER
# ============================================================
def solve_TIFA(f_E, phi_i_ratio=0.377 * np.pi):
    """
    Solve Klein-Gordon for TIFA axion.
    Returns w0, wa from CPL fit.
    """
    phi_i  = phi_i_ratio * f_E
    dphi_i = 0.0

    # Fix Lambda^4 from rho_DE(0)
    rho_DE0 = 3.0 * OmegaDE
    cos_val = np.cos(phi_i / f_E)
    denom = 1.0 - cos_val

    if abs(denom) < 1e-10:
        print(f"  WARNING: phi_i/f_E too small")
        return None

    Lambda4 = rho_DE0 / denom

    def V(phi):
        return Lambda4 * (1.0 - np.cos(phi / f_E))

    def dVdphi(phi):
        return (Lambda4 / f_E) * np.sin(phi / f_E)

    # ODE in scale factor a
    def odes(a, y):
        phi, dphi = y

        # LCDM background for KG equation
        H2 = OmegaM / a**3 + OmegaDE
        H  = np.sqrt(max(H2, 1e-30))
        dH2_da = -3.0 * OmegaM / a**4
        dH_da  = dH2_da / (2.0 * H)

        coeff  = 3.0 / a + dH_da / H
        force  = dVdphi(phi) / (H**2 * a**2)
        d2phi  = -coeff * dphi - force

        return [dphi, d2phi]

    a_start = 0.01
    a_end   = 1.0

    sol = solve_ivp(
        odes,
        [a_start, a_end],
        [phi_i, dphi_i],
        method='RK45',
        dense_output=True,
        rtol=1e-10,
        atol=1e-12,
        max_step=0.005
    )

    if not sol.success:
        print(f"  ODE failed: {sol.message}")
        return None

    # Compute w(a)
    a_arr    = np.linspace(a_start, a_end, 1000)
    phi_arr  = sol.sol(a_arr)[0]
    dphi_arr = sol.sol(a_arr)[1]

    H_arr       = np.sqrt(OmegaM / a_arr**3 + OmegaDE)
    dphi_dt_arr = H_arr * a_arr * dphi_arr
    KE_arr      = 0.5 * dphi_dt_arr**2
    V_arr       = V(phi_arr)
    denom_arr   = KE_arr + V_arr
    denom_arr   = np.where(np.abs(denom_arr) < 1e-30,
                           1e-30, denom_arr)
    w_arr       = (KE_arr - V_arr) / denom_arr

    # CPL fit over z < 2 (a > 0.333)
    mask  = a_arr > 0.333
    a_fit = a_arr[mask]
    w_fit = w_arr[mask]

    try:
        popt, _ = curve_fit(
            lambda a, w0, wa: w0 + wa*(1-a),
            a_fit, w_fit,
            p0=[-0.9, -0.3]
        )
        w0_fit = popt[0]
        wa_fit = popt[1]
    except Exception as e:
        print(f"  CPL fit failed: {e}")
        return None

    return {
        'w0'       : w0_fit,
        'wa'       : wa_fit,
        'w_today'  : w_arr[-1],
        'phi_today': phi_arr[-1],
        'phi_i'    : phi_i,
        'Lambda4'  : Lambda4
    }

# ============================================================
# MAIN
# ============================================================
def main():

    print("=" * 62)
    print("TIFA DIAGNOSTIC V2: Three-f Test")
    print("phi_i = 0.377 * pi * f_E (scales with f)")
    print(f"DH0 = c/H0 = {DH0:.2f} Mpc")
    print("=" * 62)

    f_values = [0.305, 1.0, 2.0]
    results  = []

    for f_E in f_values:
        print(f"\nRunning f_E = {f_E} M_Pl ...")

        out = solve_TIFA(f_E)
        if out is None:
            print("  FAILED")
            continue

        w0 = out['w0']
        wa = out['wa']

        print(f"  Lambda^4/rho_DE0 = {out['Lambda4']/(3*OmegaDE):.4f}")
        print(f"  phi_i            = {out['phi_i']:.4f} M_Pl")
        print(f"  phi_today        = {out['phi_today']:.4f} M_Pl")
        print(f"  w(z=0)           = {out['w_today']:.4f}")
        print(f"  w0 (CPL)         = {w0:.4f}")
        print(f"  wa (CPL)         = {wa:.4f}")

        chi2 = compute_chi2(w0, wa)
        print(f"  chi2             = {chi2:.4f}")

        results.append({
            'f_E' : f_E,
            'w0'  : w0,
            'wa'  : wa,
            'chi2': chi2
        })

    # LCDM
    print(f"\nRunning LCDM (w0=-1, wa=0) ...")
    chi2_lcdm = compute_chi2(-1.0, 0.0)
    print(f"  chi2 (LCDM) = {chi2_lcdm:.4f}")

    # Sanity check LCDM
    print(f"\nSANITY CHECK:")
    print(f"  Expected chi2(LCDM) ~ 13-25")
    print(f"  Got chi2(LCDM)      = {chi2_lcdm:.4f}")
    if chi2_lcdm < 100:
        print(f"  Units: CORRECT")
    else:
        print(f"  Units: STILL WRONG - see note below")

    # Summary
    print("\n" + "=" * 62)
    print("SUMMARY TABLE")
    print("=" * 62)
    print(f"{'f_E':>6} {'w0':>8} {'wa':>8} "
          f"{'chi2':>10} {'Δchi2':>10} {'ΔAIC':>10}")
    print("-" * 62)

    for r in results:
        dc   = r['chi2'] - chi2_lcdm
        daic = (r['chi2'] + 2) - chi2_lcdm
        print(f"{r['f_E']:>6.3f} "
              f"{r['w0']:>8.4f} "
              f"{r['wa']:>8.4f} "
              f"{r['chi2']:>10.3f} "
              f"{dc:>+10.3f} "
              f"{daic:>+10.3f}")

    print(f"{'LCDM':>6} "
          f"{-1.0:>8.4f} "
          f"{0.0:>8.4f} "
          f"{chi2_lcdm:>10.3f} "
          f"{'0.000':>10} "
          f"{'0.000':>10}")

    print("=" * 62)

    # DESI comparison
    print("\nDESI DR1 PREFERRED:")
    print("  w0 = -0.827 ± 0.060")
    print("  wa = -0.75  +0.29/-0.25")
    print("\nTIFA f_E=2.0 GIVES:")
    for r in results:
        if abs(r['f_E'] - 2.0) < 0.01:
            print(f"  w0 = {r['w0']:.4f}")
            print(f"  wa = {r['wa']:.4f}")

if __name__ == "__main__":
    main()

TIFA DIAGNOSTIC V2: Three-f Test
phi_i = 0.377 * pi * f_E (scales with f)
DH0 = c/H0 = 4450.60 Mpc

Running f_E = 0.305 M_Pl ...
  Lambda^4/rho_DE0 = 1.6048
  phi_i            = 0.3612 M_Pl
  phi_today        = -0.0323 M_Pl
  w(z=0)           = 0.7469
  w0 (CPL)         = -0.2479
  wa (CPL)         = 0.7961
  chi2             = 1933.3762

Running f_E = 1.0 M_Pl ...
  Lambda^4/rho_DE0 = 1.6048
  phi_i            = 1.1844 M_Pl
  phi_today        = 0.6742 M_Pl
  w(z=0)           = -0.2785
  w0 (CPL)         = -0.3329
  wa (CPL)         = -1.0423
  chi2             = 8156.5422

Running f_E = 2.0 M_Pl ...
  Lambda^4/rho_DE0 = 1.6048
  phi_i            = 2.3688 M_Pl
  phi_today        = 2.0966 M_Pl
  w(z=0)           = -0.8452
  w0 (CPL)         = -0.8528
  wa (CPL)         = -0.2243
  chi2             = 11101.4984

Running LCDM (w0=-1, wa=0) ...
  chi2 (LCDM) = 11887.5168

SANITY CHECK:
  Expected chi2(LCDM) ~ 13-25
  Got chi2(LCDM)      = 11887.5168
  Units: STILL WRONG - see note below

S

In [2]:

"""
TIFA Diagnostic V3
Correct BAO chi2 using
dimensionless ratios only
"""

import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import curve_fit, minimize

# ============================================================
# PARAMETERS
# ============================================================
OmegaM  = 0.315
OmegaDE = 0.685

# DESI DR1 BAO DATA
# We fit for h*rd as a combined parameter
# This removes the H0/rd degeneracy
# obs = D_M(z)*H0/c / rd_fid or DH*H0/c / rd_fid
# Simplest: work in units where c/H0 = 1
# Then D_M is in units of c/H0
# And D_H = 1/H(z) in same units
# rd in same units = rd_Mpc / (c/H0_Mpc)
# rd_Mpc = 147.09, c/H0 = 4450.6
# rd_dimless = 147.09 / 4450.6 = 0.03305

c_kms  = 299792.458
H0     = 67.36
DH0    = c_kms / H0       # 4450.6 Mpc
rd     = 147.09            # Mpc (Planck)
rd_dless = rd / DH0       # dimensionless = 0.03305

print(f"rd dimensionless = {rd_dless:.5f}")
print(f"Expected DM/rd at z=0.295 for LCDM:")
# Quick estimate
print(f"  DM ~ 0.25 * DH0 = {0.25*DH0:.1f} Mpc")
print(f"  DM/rd ~ {0.25*DH0/rd:.2f}")
print(f"  DESI  = 7.93")

# DESI data in PHYSICAL Mpc / rd
DESI_data = [
    [0.295,   7.93,  0.15,  'DM'],
    [0.295,  20.08,  0.60,  'DH'],
    [0.510,  13.62,  0.25,  'DM'],
    [0.510,  13.11,  0.42,  'DH'],
    [0.706,  16.85,  0.32,  'DM'],
    [0.706,  10.34,  0.28,  'DH'],
    [0.930,  21.71,  0.28,  'DM'],
    [0.930,   8.00,  0.17,  'DH'],
    [1.317,  27.79,  0.69,  'DM'],
    [1.317,   6.13,  0.13,  'DH'],
    [1.491,  30.21,  0.79,  'DM'],
    [1.491,   5.65,  0.13,  'DH'],
    [2.330,  39.71,  0.94,  'DM'],
]

# ============================================================
# H(z) for CPL
# ============================================================
def E(z, w0, wa):
    """H(z)/H0"""
    a = 1.0/(1.0+z)
    # Dark energy density
    # rho_de/rho_de0 = a^{-3(1+w0+wa)} exp(-3wa(1-a))
    exponent = -3.0*(1.0+w0+wa)*np.log(a) - 3.0*wa*(1.0-a)
    rho_de = OmegaDE * np.exp(exponent)
    H2 = OmegaM*(1+z)**3 + rho_de
    return np.sqrt(max(H2, 1e-30))

# ============================================================
# COMOVING DISTANCE (dimensionless, units of c/H0)
# ============================================================
def DC_dimless(z, w0, wa):
    """D_C in units of c/H0"""
    integrand = lambda zp: 1.0/E(zp, w0, wa)
    result, _ = quad(integrand, 0, z,
                     limit=200, epsabs=1e-10, epsrel=1e-10)
    return result

# ============================================================
# CHI2 - marginalised over rd×H0
# We fit rd as a free parameter to remove degeneracy
# OR fix rd_dless and compute chi2
# ============================================================
def compute_chi2(w0, wa, rd_over_DH0=None):
    """
    Compute chi2.
    rd_over_DH0: if None, marginalize over it analytically.
    """
    if rd_over_DH0 is None:
        # Marginalise over rd analytically
        # chi2 = sum [(obs - theory/rd)^2 / sigma^2]
        # where theory is in Mpc and rd is free
        # This is a 1D quadratic minimisation
        # Best rd minimises chi2
        # Find best rd then compute chi2

        # Collect numerator and denominator
        A = 0.0  # sum theory^2/sigma^2
        B = 0.0  # sum obs*theory/sigma^2
        C = 0.0  # sum obs^2/sigma^2

        for row in DESI_data:
            z, obs, sigma, dtype = row
            if dtype == 'DM':
                theory_dimless = DC_dimless(z, w0, wa)
                # D_M_Mpc = theory_dimless * DH0
                # D_M/rd = theory_dimless * DH0 / rd
                # But rd is what we marginalise over
                # Let x = 1/rd_dimless (where rd_dimless = rd/DH0)
                # theory_ratio = theory_dimless * x
                # chi2 term = (obs - theory_dimless*x)^2/sigma^2
                t = theory_dimless
            else:  # DH
                Hz = E(z, w0, wa)
                DH_dimless = 1.0/Hz
                t = DH_dimless

            A += t**2 / sigma**2
            B += obs * t / sigma**2
            C += obs**2 / sigma**2

        # Best x = B/A
        x_best = B/A
        chi2_min = C - B**2/A

        rd_best_dimless = 1.0/x_best
        rd_best_Mpc = rd_best_dimless * DH0

        return chi2_min, rd_best_Mpc

    else:
        # Fixed rd
        chi2 = 0.0
        for row in DESI_data:
            z, obs, sigma, dtype = row
            if dtype == 'DM':
                DC = DC_dimless(z, w0, wa)
                theory = DC / rd_over_DH0
            else:
                Hz = E(z, w0, wa)
                DH = 1.0/Hz
                theory = DH / rd_over_DH0
            chi2 += ((obs - theory)/sigma)**2
        return chi2, rd_over_DH0 * DH0

# ============================================================
# TIFA SOLVER
# ============================================================
def solve_TIFA(f_E, phi_i_ratio=0.377*np.pi):
    phi_i  = phi_i_ratio * f_E
    rho_DE0 = 3.0 * OmegaDE
    cos_val = np.cos(phi_i/f_E)
    denom = 1.0 - cos_val
    if abs(denom) < 1e-10:
        return None
    Lambda4 = rho_DE0 / denom

    def dVdphi(phi):
        return (Lambda4/f_E)*np.sin(phi/f_E)

    def V(phi):
        return Lambda4*(1.0 - np.cos(phi/f_E))

    def odes(a, y):
        phi, dphi = y
        H2 = OmegaM/a**3 + OmegaDE
        H  = np.sqrt(max(H2, 1e-30))
        dH2_da = -3.0*OmegaM/a**4
        dH_da  = dH2_da/(2.0*H)
        coeff  = 3.0/a + dH_da/H
        force  = dVdphi(phi)/(H**2*a**2)
        return [dphi, -coeff*dphi - force]

    sol = solve_ivp(odes, [0.01, 1.0], [phi_i, 0.0],
                   method='RK45', dense_output=True,
                   rtol=1e-10, atol=1e-12,
                   max_step=0.005)
    if not sol.success:
        return None

    a_arr    = np.linspace(0.01, 1.0, 1000)
    phi_arr  = sol.sol(a_arr)[0]
    dphi_arr = sol.sol(a_arr)[1]
    H_arr    = np.sqrt(OmegaM/a_arr**3 + OmegaDE)
    dphi_dt  = H_arr*a_arr*dphi_arr
    KE       = 0.5*dphi_dt**2
    Vp       = V(phi_arr)
    w_arr    = (KE - Vp)/(KE + Vp + 1e-30)

    mask  = a_arr > 0.333
    try:
        popt, _ = curve_fit(
            lambda a,w0,wa: w0+wa*(1-a),
            a_arr[mask], w_arr[mask],
            p0=[-0.9,-0.3])
        return {'w0':popt[0], 'wa':popt[1],
                'w_today':w_arr[-1],
                'phi_today':phi_arr[-1],
                'phi_i':phi_i}
    except:
        return None

# ============================================================
# MAIN
# ============================================================
def main():

    print("="*62)
    print("TIFA DIAGNOSTIC V3")
    print("Marginalised over rd to remove H0/rd degeneracy")
    print("="*62)

    f_values = [0.305, 1.0, 2.0]
    results  = []

    for f_E in f_values:
        print(f"\nf_E = {f_E} M_Pl")
        out = solve_TIFA(f_E)
        if out is None:
            print("  FAILED")
            continue
        w0 = out['w0']
        wa = out['wa']
        print(f"  phi_today = {out['phi_today']:.4f} M_Pl")
        print(f"  w(z=0)    = {out['w_today']:.4f}")
        print(f"  w0        = {w0:.4f}")
        print(f"  wa        = {wa:.4f}")

        chi2, rd_fit = compute_chi2(w0, wa)
        print(f"  chi2 (marginalised) = {chi2:.4f}")
        print(f"  best-fit rd         = {rd_fit:.2f} Mpc")
        results.append({'f_E':f_E,'w0':w0,'wa':wa,
                       'chi2':chi2,'rd':rd_fit})

    print(f"\nLCDM (w0=-1, wa=0)")
    chi2_lcdm, rd_lcdm = compute_chi2(-1.0, 0.0)
    print(f"  chi2 (marginalised) = {chi2_lcdm:.4f}")
    print(f"  best-fit rd         = {rd_lcdm:.2f} Mpc")

    print("\n"+"="*62)
    print("SUMMARY")
    print("="*62)
    print(f"{'f_E':>6} {'w0':>8} {'wa':>8} "
          f"{'chi2':>8} {'Δchi2':>8} {'rd_fit':>8}")
    print("-"*62)
    for r in results:
        dc = r['chi2'] - chi2_lcdm
        print(f"{r['f_E']:>6.3f} "
              f"{r['w0']:>8.4f} "
              f"{r['wa']:>8.4f} "
              f"{r['chi2']:>8.3f} "
              f"{dc:>+8.3f} "
              f"{r['rd']:>8.2f}")
    print(f"{'LCDM':>6} "
          f"{-1.0:>8.4f} "
          f"{0.0:>8.4f} "
          f"{chi2_lcdm:>8.3f} "
          f"{'0.000':>8} "
          f"{rd_lcdm:>8.2f}")

    print("\nSANITY CHECK:")
    print(f"  chi2(LCDM) = {chi2_lcdm:.3f}")
    print(f"  13 data points, 1 marginalised param")
    print(f"  Expected chi2 ~ 10-20 for good fit")
    if chi2_lcdm < 50:
        print(f"  STATUS: CORRECT")
    else:
        print(f"  STATUS: STILL WRONG")

    print("\nDESI PREFERRED:")
    print("  w0 = -0.827 ± 0.060")
    print("  wa = -0.75  +0.29/-0.25")

if __name__ == "__main__":
    main()

rd dimensionless = 0.03305
Expected DM/rd at z=0.295 for LCDM:
  DM ~ 0.25 * DH0 = 1112.7 Mpc
  DM/rd ~ 7.56
  DESI  = 7.93
TIFA DIAGNOSTIC V3
Marginalised over rd to remove H0/rd degeneracy

f_E = 0.305 M_Pl
  phi_today = -0.0323 M_Pl
  w(z=0)    = 0.7469
  w0        = -0.2479
  wa        = 0.7961
  chi2 (marginalised) = 1820.7462
  best-fit rd         = 156.52 Mpc

f_E = 1.0 M_Pl
  phi_today = 0.6742 M_Pl
  w(z=0)    = -0.2785
  w0        = -0.3329
  wa        = -1.0423
  chi2 (marginalised) = 3771.5258
  best-fit rd         = 208.17 Mpc

f_E = 2.0 M_Pl
  phi_today = 2.0966 M_Pl
  w(z=0)    = -0.8452
  w0        = -0.8528
  wa        = -0.2243
  chi2 (marginalised) = 3791.4032
  best-fit rd         = 225.98 Mpc

LCDM (w0=-1, wa=0)
  chi2 (marginalised) = 3772.5737
  best-fit rd         = 230.18 Mpc

SUMMARY
   f_E       w0       wa     chi2    Δchi2   rd_fit
--------------------------------------------------------------
 0.305  -0.2479   0.7961 1820.746 -1951.828   156.52
 1.000  -0.

In [4]:

"""
TIFA Diagnostic V4
Use exact w(a) from field solver for distances
Only use CPL for reporting w0, wa
"""

import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d

# ============================================================
# PARAMETERS
# ============================================================
OmegaM  = 0.315
OmegaDE = 0.685
c_kms   = 299792.458
H0      = 67.36
DH0     = c_kms / H0
rd      = 147.09

DESI_data = [
    [0.295,   7.93,  0.15,  'DM'],
    [0.295,  20.08,  0.60,  'DH'],
    [0.510,  13.62,  0.25,  'DM'],
    [0.510,  13.11,  0.42,  'DH'],
    [0.706,  16.85,  0.32,  'DM'],
    [0.706,  10.34,  0.28,  'DH'],
    [0.930,  21.71,  0.28,  'DM'],
    [0.930,   8.00,  0.17,  'DH'],
    [1.317,  27.79,  0.69,  'DM'],
    [1.317,   6.13,  0.13,  'DH'],
    [1.491,  30.21,  0.79,  'DM'],
    [1.491,   5.65,  0.13,  'DH'],
    [2.330,  39.71,  0.94,  'DM']
]

In [5]:

"""
TIFA Diagnostic V4
Definitive version.
Uses exact w(a) interpolation.
No CPL for distances.
"""

import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d

# ============================================================
# PARAMETERS
# ============================================================
OmegaM  = 0.315
OmegaDE = 0.685
c_kms   = 299792.458
H0      = 67.36
DH0     = c_kms / H0      # 4450.6 Mpc
rd      = 147.09           # Mpc

# ============================================================
# DESI DR1 BAO
# ============================================================
DESI_data = [
    [0.295,   7.93,  0.15,  'DM'],
    [0.295,  20.08,  0.60,  'DH'],
    [0.510,  13.62,  0.25,  'DM'],
    [0.510,  13.11,  0.42,  'DH'],
    [0.706,  16.85,  0.32,  'DM'],
    [0.706,  10.34,  0.28,  'DH'],
    [0.930,  21.71,  0.28,  'DM'],
    [0.930,   8.00,  0.17,  'DH'],
    [1.317,  27.79,  0.69,  'DM'],
    [1.317,   6.13,  0.13,  'DH'],
    [1.491,  30.21,  0.79,  'DM'],
    [1.491,   5.65,  0.13,  'DH'],
    [2.330,  39.71,  0.94,  'DM'],
]

# ============================================================
# SANITY CHECK ON DATA
# Print what LCDM predicts for
# each data point before any
# TIFA calculation
# ============================================================
def lcdm_E(z):
    """H(z)/H0 for LCDM"""
    return np.sqrt(OmegaM*(1+z)**3 + OmegaDE)

def lcdm_DM(z):
    """D_M in Mpc for LCDM"""
    integrand = lambda zp: 1.0/lcdm_E(zp)
    result, _ = quad(integrand, 0, z,
                     limit=200, epsabs=1e-10)
    return result * DH0

def lcdm_DH(z):
    """D_H = c/H(z) in Mpc for LCDM"""
    return DH0 / lcdm_E(z)

def print_sanity_check():
    print("\nSANITY CHECK: LCDM predictions vs DESI data")
    print(f"{'z':>6} {'type':>4} {'LCDM':>8} "
          f"{'DESI':>8} {'diff/sigma':>10}")
    print("-"*45)
    for row in DESI_data:
        z, obs, sigma, dtype = row
        if dtype == 'DM':
            theory = lcdm_DM(z) / rd
        else:
            theory = lcdm_DH(z) / rd
        pull = (obs - theory)/sigma
        print(f"{z:>6.3f} {dtype:>4} {theory:>8.3f} "
              f"{obs:>8.3f} {pull:>+10.3f}")

# ============================================================
# TIFA FIELD SOLVER
# Returns interpolated w(a) function
# ============================================================
def solve_TIFA_exact(f_E, phi_i_ratio=0.377*np.pi):
    """
    Solve KG equation.
    Returns dict with w_interp function
    and other diagnostics.
    """
    phi_i   = phi_i_ratio * f_E
    rho_DE0 = 3.0 * OmegaDE
    cos_val = np.cos(phi_i/f_E)
    denom   = 1.0 - cos_val

    if abs(denom) < 1e-10:
        return None

    Lambda4 = rho_DE0 / denom

    def V(phi):
        return Lambda4*(1.0 - np.cos(phi/f_E))

    def dVdphi(phi):
        return (Lambda4/f_E)*np.sin(phi/f_E)

    def odes(a, y):
        phi, dphi = y
        # Background: LCDM (phi contribution small)
        H2     = OmegaM/a**3 + OmegaDE
        H      = np.sqrt(max(H2, 1e-30))
        dH_da  = -3.0*OmegaM/(4.0*a**4*H)
        coeff  = 3.0/a + dH_da/H
        force  = dVdphi(phi)/(H**2*a**2)
        return [dphi, -coeff*dphi - force]

    sol = solve_ivp(
        odes,
        [0.001, 1.0],
        [phi_i, 0.0],
        method='DOP853',
        dense_output=True,
        rtol=1e-12,
        atol=1e-14,
        max_step=0.002
    )

    if not sol.success:
        print(f"  ODE failed: {sol.message}")
        return None

    # Dense output on fine grid
    a_arr    = np.linspace(0.001, 1.0, 5000)
    phi_arr  = sol.sol(a_arr)[0]
    dphi_arr = sol.sol(a_arr)[1]

    H_arr    = np.sqrt(OmegaM/a_arr**3 + OmegaDE)
    dphi_dt  = H_arr * a_arr * dphi_arr
    KE       = 0.5 * dphi_dt**2
    Vp       = V(phi_arr)
    total    = KE + Vp
    total    = np.where(np.abs(total)<1e-30, 1e-30, total)
    w_arr    = (KE - Vp) / total

    # Clamp w to physical range
    w_arr = np.clip(w_arr, -2.0, 1.0)

    # Build interpolator w(a)
    w_interp = interp1d(
        a_arr, w_arr,
        kind='cubic',
        bounds_error=False,
        fill_value=(w_arr[0], w_arr[-1])
    )

    # CPL fit for reporting only
    mask = a_arr > 0.333
    try:
        popt, _ = curve_fit(
            lambda a,w0,wa: w0 + wa*(1-a),
            a_arr[mask], w_arr[mask],
            p0=[-0.9, -0.3]
        )
        w0_cpl = popt[0]
        wa_cpl = popt[1]
    except:
        w0_cpl = w_arr[-1]
        wa_cpl = 0.0

    return {
        'w_interp' : w_interp,
        'w0_cpl'   : w0_cpl,
        'wa_cpl'   : wa_cpl,
        'w_today'  : w_arr[-1],
        'phi_today': phi_arr[-1],
        'phi_i'    : phi_i,
        'Lambda4'  : Lambda4,
        'a_arr'    : a_arr,
        'w_arr'    : w_arr
    }

# ============================================================
# HUBBLE USING EXACT w(a)
# ============================================================
def E_exact(z, w_interp):
    """
    H(z)/H0 using exact w(a) from field solver.
    """
    a = 1.0/(1.0+z)

    def integrand(ap):
        return 3.0*(1.0 + w_interp(ap))/ap

    # Integrate from a to 1
    if abs(a - 1.0) < 1e-10:
        rho_de_ratio = 1.0
    else:
        val, _ = quad(integrand, a, 1.0,
                      limit=500,
                      epsabs=1e-12,
                      epsrel=1e-12)
        rho_de_ratio = np.exp(val)

    H2 = OmegaM*(1.0+z)**3 + OmegaDE*rho_de_ratio
    return np.sqrt(max(H2, 1e-30))

# ============================================================
# DISTANCES USING EXACT w(a)
# ============================================================
def DM_exact(z, w_interp):
    """D_M in Mpc using exact w(a)"""
    integrand = lambda zp: 1.0/E_exact(zp, w_interp)
    result, _ = quad(integrand, 0, z,
                     limit=500,
                     epsabs=1e-12,
                     epsrel=1e-12)
    return result * DH0

def DH_exact(z, w_interp):
    """D_H = c/H(z) in Mpc"""
    return DH0 / E_exact(z, w_interp)

# ============================================================
# CHI2 USING EXACT w(a)
# ============================================================
def compute_chi2_exact(w_interp):
    """Chi2 using exact w(a) interpolation."""
    chi2 = 0.0
    for row in DESI_data:
        z, obs, sigma, dtype = row
        if dtype == 'DM':
            theory = DM_exact(z, w_interp) / rd
        else:
            theory = DH_exact(z, w_interp) / rd
        chi2 += ((obs - theory)/sigma)**2
    return chi2

# ============================================================
# LCDM CHI2
# ============================================================
def compute_chi2_lcdm():
    chi2 = 0.0
    for row in DESI_data:
        z, obs, sigma, dtype = row
        if dtype == 'DM':
            theory = lcdm_DM(z)/rd
        else:
            theory = lcdm_DH(z)/rd
        chi2 += ((obs-theory)/sigma)**2
    return chi2

# ============================================================
# MAIN
# ============================================================
def main():

    print("="*62)
    print("TIFA DIAGNOSTIC V4")
    print("Exact w(a) used for all distance calculations")
    print("CPL used for reporting only")
    print("="*62)

    # Sanity check first
    print_sanity_check()

    chi2_lcdm = compute_chi2_lcdm()
    print(f"\nchi2(LCDM) = {chi2_lcdm:.4f}")
    print(f"Expected   ~ 13-25")
    if chi2_lcdm < 50:
        print(f"STATUS: CORRECT")
    else:
        print(f"STATUS: ISSUE WITH DATA OR PARAMETERS")
        print(f"Check the sanity table above.")
        print(f"Large pulls indicate H0/rd mismatch.")

    f_values = [0.305, 1.0, 2.0]
    results  = []

    print(f"\n{'='*62}")
    for f_E in f_values:
        print(f"\nf_E = {f_E} M_Pl")
        out = solve_TIFA_exact(f_E)
        if out is None:
            print("  FAILED")
            continue

        print(f"  phi_i     = {out['phi_i']:.4f} M_Pl")
        print(f"  phi_today = {out['phi_today']:.4f} M_Pl")
        print(f"  w(z=0)    = {out['w_today']:.4f}")
        print(f"  w0 (CPL)  = {out['w0_cpl']:.4f}")
        print(f"  wa (CPL)  = {out['wa_cpl']:.4f}")

        chi2 = compute_chi2_exact(out['w_interp'])
        print(f"  chi2      = {chi2:.4f}")

        results.append({
            'f_E' : f_E,
            'w0'  : out['w0_cpl'],
            'wa'  : out['wa_cpl'],
            'chi2': chi2
        })

    print(f"\n{'='*62}")
    print("SUMMARY")
    print(f"{'='*62}")
    print(f"{'Model':>8} {'w0':>8} {'wa':>8} "
          f"{'chi2':>10} {'Δchi2':>10} {'ΔAIC':>10}")
    print("-"*62)
    for r in results:
        dc   = r['chi2'] - chi2_lcdm
        daic = dc + 2
        print(f"{r['f_E']:>8.3f} "
              f"{r['w0']:>8.4f} "
              f"{r['wa']:>8.4f} "
              f"{r['chi2']:>10.4f} "
              f"{dc:>+10.4f} "
              f"{daic:>+10.4f}")
    print(f"{'LCDM':>8} "
          f"{-1.0:>8.4f} "
          f"{0.0:>8.4f} "
          f"{chi2_lcdm:>10.4f} "
          f"{'0':>10} "
          f"{'0':>10}")

    print(f"\nDESI PREFERRED:")
    print(f"  w0 = -0.827 ± 0.060")
    print(f"  wa = -0.75  +0.29/-0.25")

    print(f"\nNOTE ON SANITY CHECK:")
    print(f"  If chi2(LCDM) > 50, the H0/rd values")
    print(f"  are not consistent with this DESI dataset.")
    print(f"  This affects absolute chi2 values but")
    print(f"  NOT the relative Δchi2 between models.")
    print(f"  Δchi2 is the scientifically meaningful number.")

if __name__ == "__main__":
    main()

TIFA DIAGNOSTIC V4
Exact w(a) used for all distance calculations
CPL used for reporting only

SANITY CHECK: LCDM predictions vs DESI data
     z type     LCDM     DESI diff/sigma
---------------------------------------------
 0.295   DM    8.282    7.930     -2.346
 0.295   DH   25.859   20.080     -9.632
 0.510   DM   13.503   13.620     +0.469
 0.510   DH   22.746   13.110    -22.943
 0.706   DM   17.704   16.850     -2.669
 0.706   DH   20.176   10.340    -35.129
 0.930   DM   21.929   21.710     -0.784
 0.930   DH   17.618    8.000    -56.577
 1.317   DM   28.033   27.790     -0.352
 1.317   DH   14.103    6.130    -61.329
 1.491   DM   30.375   30.210     -0.208
 1.491   DH   12.839    5.650    -55.301
 2.330   DM   39.197   39.710     +0.546

chi2(LCDM) = 11887.5168
Expected   ~ 13-25
STATUS: ISSUE WITH DATA OR PARAMETERS
Check the sanity table above.
Large pulls indicate H0/rd mismatch.


f_E = 0.305 M_Pl
  phi_i     = 0.3612 M_Pl
  phi_today = -0.0468 M_Pl
  w(z=0)    = 0.2772
